In [4]:
import geopandas as gpd
import pandas as pd

df_cbsa = gpd.read_file(
    "../data/raw/tl_2024_us_cbsa.zip"
)

df_metdiv = gpd.read_file(
    "../data/raw/tl_2024_us_metdiv.zip"
)

In [5]:
print(df_cbsa.columns)
print("-------------")
print(df_metdiv.columns)


Index(['CSAFP', 'CBSAFP', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'LSAD',
       'MEMI', 'MTFCC', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')
-------------
Index(['CSAFP', 'CBSAFP', 'METDIVFP', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'MTFCC', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


In [6]:
df_cbsa.isnull().sum()

CSAFP       353
CBSAFP        0
GEOID         0
GEOIDFQ       0
NAME          0
NAMELSAD      0
LSAD          0
MEMI          0
MTFCC         0
ALAND         0
AWATER        0
INTPTLAT      0
INTPTLON      0
geometry      0
dtype: int64

In [7]:
df_metdiv.isnull().sum()

CSAFP       2
CBSAFP      0
METDIVFP    0
GEOID       0
GEOIDFQ     0
NAME        0
NAMELSAD    0
LSAD        0
MTFCC       0
ALAND       0
AWATER      0
INTPTLAT    0
INTPTLON    0
geometry    0
dtype: int64

In [8]:
cbsa_coordinates = df_cbsa[
    [
        "GEOID",
        "INTPTLAT",
        "INTPTLON"
    ]
].copy()

cbsa_coordinates = cbsa_coordinates.rename(columns={
    "GEOID": "geo_id",
    "INTPTLAT": "latitude",
    "INTPTLON": "longitude"
})

cbsa_coordinates["geography_type"] = "metro"

In [9]:
division_coordinates = df_metdiv[
    [
        "METDIVFP",
        "INTPTLAT",
        "INTPTLON"
    ]
].copy()

division_coordinates = division_coordinates.rename(columns={
    "METDIVFP": "geo_id",
    "INTPTLAT": "latitude",
    "INTPTLON": "longitude"
})

division_coordinates["geography_type"] = "division"

In [10]:
for df in [
    cbsa_coordinates,
    division_coordinates
]:
    df["latitude"] = pd.to_numeric(
        df["latitude"],
        errors="coerce"
    )

    df["longitude"] = pd.to_numeric(
        df["longitude"],
        errors="coerce"
    )

In [11]:
geography_coordinates = pd.concat([cbsa_coordinates,division_coordinates],ignore_index=True)

In [12]:
geography_coordinates.head()

print(geography_coordinates.isna().sum())

geo_id            0
latitude          0
longitude         0
geography_type    0
dtype: int64


In [13]:
df_bridge = pd.read_csv("../data/processed/geography_bridge.csv",dtype={"geo_id": str})

In [14]:
geography_coordinates["geo_id"] = (geography_coordinates["geo_id"].astype(str))

In [15]:
df_bridge_test = df_bridge.merge(
    geography_coordinates,
    on=["geo_id","geography_type"],
    how="left",
    validate="one_to_one"
)

In [16]:
print("Bridge markets:",len(df_bridge_test))

print("Missing latitude:",df_bridge_test["latitude"].isna().sum())

print("Missing longitude:",df_bridge_test["longitude"].isna().sum())

Bridge markets: 377


KeyError: 'latitude'

In [18]:
df_bridge_test=pd.read_csv("../data/processed/geography_bridge.csv")
print(len(df_bridge_test))


377
